In [ ]:
!pip install torcheval onnxruntime onnxscript

In [3]:
import logging
logging.basicConfig()
lg = logging.getLogger()
[lg.removeHandler(h) for h in lg.handlers]
lg.addHandler(logging.StreamHandler())
lg.setLevel(logging.INFO)
lg.handlers[0].setFormatter(logging.Formatter("%(asctime)s.%(msecs)03d %(pathname)s:%(lineno)d %(message)s", datefmt="%Y-%m-%d %H:%M:%S"))

import os
import numpy as np

In [4]:
def download():
  os.system("cp drive/MyDrive/tmp/data.tar.gz .")
  os.system("rm -r data")
  os.system("tar zxf data.tar.gz")
  os.system("rm data.tar.gz")


#download()

In [5]:
!ls -l data

total 20287868
-rw-rw-r-- 1 1000 1000 560000000 Mar 27 16:32 action_00
-rw-rw-r-- 1 1000 1000 560000000 Mar 27 16:32 action_01
-rw-rw-r-- 1 1000 1000 560000000 Mar 27 16:32 action_02
-rw-rw-r-- 1 1000 1000 560000000 Mar 27 16:32 action_03
-rw-rw-r-- 1 1000 1000 560000000 Mar 27 16:32 action_04
-rw-rw-r-- 1 1000 1000 560000000 Mar 27 16:32 action_05
-rw-rw-r-- 1 1000 1000 560000000 Mar 27 16:32 action_06
-rw-rw-r-- 1 1000 1000 560000000 Mar 27 16:32 action_07
-rw-rw-r-- 1 1000 1000 560000000 Mar 27 16:32 action_08
-rw-rw-r-- 1 1000 1000 560000000 Mar 27 16:32 action_09
-rw-rw-r-- 1 1000 1000 560000000 Mar 27 16:32 action_10
-rw-rw-r-- 1 1000 1000 560000000 Mar 27 16:32 action_11
-rw-rw-r-- 1 1000 1000 560000000 Mar 27 16:32 action_12
-rw-rw-r-- 1 1000 1000 560000000 Mar 27 16:32 action_13
-rw-rw-r-- 1 1000 1000 560000000 Mar 27 16:32 action_14
-rw-rw-r-- 1 1000 1000 560000000 Mar 27 16:32 action_15
-rw-rw-r-- 1 1000 1000 560000000 Mar 27 16:32 action_16
-rw-rw-r-- 1 1000 1000 560000000 

In [ ]:
!python train.py -data_dir=data -run_dir=drive/MyDrive/tmp/runs/gsolve

2026-03-31 01:14:18.963329: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774919659.406388    4432 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774919659.529472    4432 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774919660.507290    4432 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774919660.507340    4432 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774919660.507345    4432 computation_placer.cc:177] computation placer alr

In [6]:
import json

import torch
import onnxruntime

import train
import util


def _load_config(run_dir):
    cfg_dir = os.path.join(run_dir, "config")
    cfg_paths = util.get_paths_desc(cfg_dir)
    last_path = cfg_paths[-1]["path"]
    with open(last_path, "r") as f:
        jstr = f.read()
    cfg_dict = json.loads(jstr)

    cfg = train.Config()
    cfg.__dict__.update(cfg_dict)
    return cfg


def write_err(wrongf, x, y):
  s = ",".join([str(a) for a in x] + [str(y)])
  wrongf.write(s+"\n")


def evaluate(wrongpath, run_dir, data_dir):
  device = torch.device("cuda:0")
  data = train.Data(data_dir)
  #data.files = [{"path": "gsolve_wrong.csv"}]

  cfg = _load_config(run_dir)
  agent = train.Agent(cfg)
  cp_dir = os.path.join(run_dir, "checkpoint")
  train._load_checkpoint(agent, cp_dir)
  model = agent.net
  model.eval()
  model.to(device)

  with open(wrongpath, "w") as wrongf:
    for dataf in data.files:
      ts = np.loadtxt(dataf["path"], dtype=np.uint8, delimiter=",")

      with torch.no_grad():
        x = torch.from_numpy(ts[:, :54]).to(device)
        y = torch.from_numpy(ts[:, 54:]).to(device)
        y = torch.squeeze(y, dim=[1])

        logits = agent.net(x.to(torch.float32))
        pred = torch.argmax(logits, dim=1)

        rtwr = (pred - y).cpu().numpy()

        del x, y, logits, pred
        torch.cuda.empty_cache()

      wrongs = 0
      for i, w in enumerate(rtwr):
        if w != 0:
          write_err(wrongf, ts[i, :54], ts[i, 54])
          wrongs += 1

      logging.info("%s %d", dataf["path"], wrongs)


evaluate("gsolve_wrong.csv", "drive/MyDrive/tmp/runs/gsolve", "data")

2026-03-31 11:27:22.139 /usr/local/lib/python3.12/dist-packages/numexpr/utils.py:164 NumExpr defaulting to 2 threads.
2026-03-31 11:27:35.856 /content/train.py:130 loaded checkpoint drive/MyDrive/tmp/runs/gsolve/checkpoint/checkpoint_005169.pth
2026-03-31 11:27:49.764 /tmp/ipykernel_2000/1632939059.py:64 data/action_00 62420
2026-03-31 11:28:02.358 /tmp/ipykernel_2000/1632939059.py:64 data/action_01 62430
2026-03-31 11:28:14.808 /tmp/ipykernel_2000/1632939059.py:64 data/action_02 63238
2026-03-31 11:28:26.754 /tmp/ipykernel_2000/1632939059.py:64 data/action_03 62501
2026-03-31 11:28:40.133 /tmp/ipykernel_2000/1632939059.py:64 data/action_04 62987
2026-03-31 11:28:53.076 /tmp/ipykernel_2000/1632939059.py:64 data/action_05 62990
2026-03-31 11:29:05.937 /tmp/ipykernel_2000/1632939059.py:64 data/action_06 62556
2026-03-31 11:29:18.946 /tmp/ipykernel_2000/1632939059.py:64 data/action_07 63473
2026-03-31 11:29:31.955 /tmp/ipykernel_2000/1632939059.py:64 data/action_08 63188
2026-03-31 11:29:

In [9]:
def boardToInt(board):
  nb = list(reversed(board))
  return int("".join([str(x) for x in nb]), 2)


def dump_wrong_map(dstpath, srcpath):
  wrongs = np.loadtxt(srcpath, dtype=np.uint8, delimiter=",")
  wm = {}
  for i in range(wrongs.shape[0]):
    board = wrongs[i, :54].tolist()
    a = int(wrongs[i, 54])
    bint = boardToInt(board)
    if bint in wm:
      raise Exception("{}".format(i))
    wm[bint] = a
  with open(dstpath, "w") as f:
    f.write(json.dumps(wm))


dstpath = "wrong_map.json"
dump_wrong_map(dstpath, "gsolve_wrong.csv")
os.system("tar zcf wrong_map.tar.gz {}".format(dstpath))

0